# Steerling-8B: Text Generation

Steerling is a **causal diffusion language model**. Unlike standard autoregressive models that
generate one token at a time left-to-right, Steerling generates text in **blocks** (default 64 tokens which is the size of steerling attention mask).

Within each block, tokens start fully masked and are iteratively **unmasked** over multiple denoising
steps. At each step, the model scores all masked positions and reveals the ones it is most confident
about first. This confidence-based ordering means generation is **not strictly left-to-right** —
the model can fill in the middle of a sentence before the end.

**Requirements:** GPU with >= 18 GB VRAM (A100, A6000, RTX 4090)

## 1. Load Model

We load the model via HuggingFace `AutoModel` and wrap it with `SteerlingGenerator`,
which handles the block-by-block denoising loop.

First run downloads ~17 GB from HuggingFace Hub; subsequent runs load from cache.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig

model_id = "guidelabs/steerling-8b"

model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")
print(generator)
print(f"\nBlock size: {generator.diff_block_size} tokens")
print(f"Interpretable: {generator.is_interpretable}")

## 2. Generate Text

`generator.generate()` takes a prompt string and a `GenerationConfig`, returns the generated text.
`generator.generate_full()` returns a `GenerationOutput` with text, tokens, and counts.

Key parameters:
- **`max_new_tokens`** — how many tokens to generate. Must be divisible by `block_length` (default 64).
- **`steps`** — total denoising steps, split evenly across blocks. More steps = more refinement per block.
- **`temperature`** — controls randomness. `0.0` is greedy (deterministic); higher values add Gumbel noise for diversity.

When `steps == max_new_tokens`, each denoising step unmasks exactly one token per block.

In [ ]:
prompt = "The key to understanding artificial intelligence is"
config = GenerationConfig(max_new_tokens=128, steps=128, temperature=0.4, seed=42)

output = generator.generate_full(prompt, config)

print(f"Prompt:    {prompt}")
print(f"Generated: {output.text}")
print(f"Tokens:    {output.generated_tokens} generated, {output.prompt_tokens} prompt")

## 3. Special Tokens and Early Stopping

Steerling uses a custom tokenizer (tiktoken cl100k_base) with 4 additional special tokens:

| Token | ID | Purpose |
|-------|------|---------|
| `<\|endofchunk\|>` | 100279 | End-of-chunk — separates conceptual segments in training data |
| `<\|mask\|>` | 100280 | Mask token used during diffusion denoising |

The **`<|endofchunk|>`** token: the model was trained to insert it
between conceptual segments of text. You can use it as a **stop token** to tell the generator
to halt after one chunk instead of filling all `gen_length` tokens.

In [ ]:
prompt = "Once upon a time"

# Without stop tokens — generates all 256 tokens
output_full = generator.generate_full(
    prompt,
    GenerationConfig(max_new_tokens=256, steps=256, temperature=0.4, seed=42),
)

# With stop tokens — stops at first <|endofchunk|>
output_stopped = generator.generate_full(
    prompt,
    GenerationConfig(
        max_new_tokens=256,
        steps=256,
        temperature=0.4,
        seed=42,
        stop_tokens=[tokenizer.endofchunk_token_id],
    ),
)

print(f"=== Full generation ({output_full.generated_tokens} tokens) ===")
print(output_full.text)
print(f"\n=== With endofchunk stop ({output_stopped.generated_tokens} tokens) ===")
print(output_stopped.text)

## Generation Parameters Reference

| Parameter          | Default              | Description                                              |
|--------------------|----------------------|----------------------------------------------------------|
| `max_new_tokens`   | 256                  | Number of tokens to generate                             |
| `steps`            | 256                  | Total denoising steps (split evenly across blocks)       |
| `temperature`      | 0.0                  | Gumbel noise temperature (0 = greedy, higher = more random) |
| `cfg_scale`        | 0.0                  | Classifier-free guidance scale (0 = disabled)            |
| `seed`             | None                 | Random seed for reproducibility                          |
| `stop_tokens`      | None                 | Token IDs that trigger early stop between blocks         |

**Constraints:**
- `max_new_tokens` must be divisible by `block_length` (default 64)
- `steps` must be divisible by the number of blocks (`max_new_tokens / block_length`)